# Sparse-to-Dense depth completion on KITTI (camera + LiDAR)

Runtime → Change runtime type → **T4 GPU** (or better).

Set `REPO_URL` below to your GitHub repo. Code is cloned to the Colab disk; checkpoints and results are
written to Google Drive (`RUNS_DIR`) so they survive a runtime reset.

Rough timing on a T4: download + unzip ~5 min, each training run is capped by `TIME_BUDGET_MIN`.

In [ ]:
!nvidia-smi

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')

REPO_URL = 'https://github.com/<your-username>/kitti-code.git'  # <-- change this
PROJECT_DIR = '/content/kitti-code'                    # cloned code (Colab disk)
DATA_DIR = '/content/data'                             # dataset (Colab disk: much faster than Drive)
RUNS_DIR = '/content/drive/MyDrive/kitti-runs'         # checkpoints + results (persist on Drive)
TIME_BUDGET_MIN = 30                                   # per training run

if os.path.isdir(PROJECT_DIR):
    !git -C {PROJECT_DIR} pull --ff-only
else:
    !git clone {REPO_URL} {PROJECT_DIR}

os.makedirs(RUNS_DIR, exist_ok=True)
%cd {PROJECT_DIR}
!pip install -q -r requirements.txt

## 1. Download the KITTI depth selection subset (~2 GB)
Requires having registered on the KITTI website and accepted its license.

In [ ]:
!bash scripts/download_data.sh {DATA_DIR}

## 2. Train the multimodal model (RGB + LiDAR)

In [ ]:
!python train.py --data_root {DATA_DIR} --modality rgbd --out_dir {RUNS_DIR}/rgbd --time_budget_min {TIME_BUDGET_MIN}

## 3. (Optional) Ablations: LiDAR only and camera only
These show how much each sensor contributes. Use `--layers 18` or a smaller `TIME_BUDGET_MIN` to stay within 2 hours total.

In [ ]:
!python train.py --data_root {DATA_DIR} --modality d --out_dir {RUNS_DIR}/d --time_budget_min {TIME_BUDGET_MIN}

In [ ]:
!python train.py --data_root {DATA_DIR} --modality rgb --out_dir {RUNS_DIR}/rgb --time_budget_min {TIME_BUDGET_MIN}

## 4. Evaluate and compare

In [ ]:
for run in ['rgbd', 'd', 'rgb']:
    ckpt = f'{RUNS_DIR}/{run}/best.pt'
    if os.path.exists(ckpt):
        !python evaluate.py --checkpoint {ckpt} --data_root {DATA_DIR}

In [ ]:
import json, glob
import pandas as pd

rows = [json.load(open(p)) for p in sorted(glob.glob(f'{RUNS_DIR}/*/eval/metrics.json'))]
df = pd.DataFrame(rows).set_index('modality')[['epoch', 'num_frames', 'rmse', 'mae', 'irmse', 'imae', 'delta1']]
df.round(3)  # RMSE/MAE in mm, iRMSE/iMAE in 1/km

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4))
for p in sorted(glob.glob(f'{RUNS_DIR}/*/history.csv')):
    h = pd.read_csv(p)
    ax.plot(h['epoch'], h['rmse'], marker='o', label=os.path.basename(os.path.dirname(p)))
ax.set_xlabel('epoch'); ax.set_ylabel('val RMSE (mm)'); ax.legend(); ax.grid(alpha=0.3)
plt.show()

In [ ]:
from IPython.display import Image, display

for p in sorted(glob.glob(f'{RUNS_DIR}/rgbd/eval/vis/*.png'))[:3]:
    display(Image(filename=p))